# Mamba-YOLO nguyên bản — chọn T để so sánh với YOLOv12s
Giữ nguyên code tác giả. Không ghép YOLOv12 vào model. `T` là bản nhỏ trong họ Mamba-YOLO, **không tương đương chính xác** với YOLOv12s về số tham số/FLOPs.

Bật Internet và GPU **T4** (chỉ dùng GPU 0). Notebook tạo Python 3.11 + PyTorch 2.3.0 riêng để phù hợp source cũ. Không dùng script mbyolo_train.py có lỗi dispatch; gọi API train trực tiếp, không sửa file source. Chưa chạy xác minh trên GPU Kaggle trong môi trường chuẩn bị notebook.


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path('/kaggle/working/Mamba-YOLO')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/mluu59990-collab/Mamba-YOLO.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['git','rev-parse','HEAD'],check=True)


## 1. Môi trường gốc, CUDA Toolkit 12.1 và GCC 11 riêng
Chọn GPU T4 và bật Internet. Cài lần đầu cần tải compiler/runtime. Không thay driver hay CUDA hệ thống. Log: /kaggle/working/selective_scan_build.log. Dừng nếu setup lỗi.


In [ ]:
import os, sys, subprocess, shutil, tarfile, re
from pathlib import Path

SETUP_OK = False
os.environ['MPLBACKEND'] = 'Agg'
LOG = Path('/kaggle/working/selective_scan_build.log')
LOG.write_text('Mamba-YOLO: isolated CUDA 12.1 + GCC 11 setup\n')

def run_logged(command, env=None):
    with LOG.open('a') as logfile:
        heading = '\n$ ' + ' '.join(map(str, command)) + '\n'
        print(heading, flush=True)
        logfile.write(heading); logfile.flush()
        process = subprocess.Popen(list(map(str, command)), env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            logfile.write(line); logfile.flush()
        if process.wait():
            raise RuntimeError(f'Command failed; see {LOG}. Do not run training yet.')

run_logged([sys.executable, '-m', 'pip', 'install', 'uv'])
ENV = Path('/kaggle/working/mamba-original-env')
if not (ENV/'bin/python').exists():
    run_logged([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', str(ENV)])
PY = str(ENV/'bin/python')
constraints = Path('/kaggle/working/mamba-constraints.txt')
constraints.write_text('torch==2.3.0\ntorchvision==0.18.0\nnumpy==1.26.4\n')
run_logged([PY, '-m', 'pip', 'install', '-c', str(constraints),
            'torch==2.3.0', 'torchvision==0.18.0', 'numpy==1.26.4',
            '--index-url', 'https://download.pytorch.org/whl/cu121'])
run_logged([PY, '-m', 'pip', 'install', '-c', str(constraints),
            'setuptools==69.5.1', 'wheel==0.43.0', 'packaging==24.0',
            'timm==0.9.16', 'einops==0.8.0', 'seaborn', 'thop', 'ninja', '-e', str(ROOT)])
run_logged([PY, '-c', "import torch; assert torch.cuda.is_available(), 'Enable GPU T4'; assert torch.cuda.get_device_capability()[0] in (7,8,9), 'Use T4 with the original source'; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name())"])

# Isolated NVIDIA toolkit; no sudo and no replacement of Kaggle's driver/CUDA 13.
TOOLS = Path('/kaggle/working/mamba-build-tools')
TOOLS.mkdir(exist_ok=True)
MAMBA = TOOLS/'micromamba'
if not MAMBA.exists():
    archive = TOOLS/'micromamba.tar.bz2'
    run_logged(['curl', '-fL', '--retry', '3', '--connect-timeout', '30',
                'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', str(archive)])
    with tarfile.open(archive, 'r:bz2') as tar:
        member = tar.getmember('bin/micromamba')
        assert member.isfile(), 'Invalid micromamba archive'
        with tar.extractfile(member) as stream:
            MAMBA.write_bytes(stream.read())
    MAMBA.chmod(0o755)
    archive.unlink()

TOOLCHAIN = TOOLS/'cuda121-gcc11'
mamba_env = dict(os.environ, MAMBA_ROOT_PREFIX=str(TOOLS/'mamba-root'))
# These CUDA 12.1 package versions are verified in NVIDIA's cuda-12.1.1 channel.
run_logged([str(MAMBA), 'create' if not (TOOLCHAIN/'conda-meta/history').exists() else 'install',
            '-y', '-p', str(TOOLCHAIN), '--override-channels',
            '-c', 'nvidia/label/cuda-12.1.1', '-c', 'conda-forge',
            'cuda-nvcc=12.1.105', 'cuda-cudart=12.1.105', 'cuda-cudart-dev=12.1.105',
            'cuda-cccl=12.1.109', 'cuda-driver-dev=12.1.105',
            'gcc_linux-64=11', 'gxx_linux-64=11'], env=mamba_env)

NVCC = TOOLCHAIN/'bin/nvcc'
CC = TOOLCHAIN/'bin/x86_64-conda-linux-gnu-cc'
CXX = TOOLCHAIN/'bin/x86_64-conda-linux-gnu-c++'
for executable in (NVCC, CC, CXX):
    assert executable.is_file(), f'Missing tool: {executable}'
assert (TOOLCHAIN/'include/cuda_runtime.h').is_file(), 'CUDA headers missing'
# All child processes, including probe/train/resume, inherit the selected toolkit.
os.environ.update(CUDA_HOME=str(TOOLCHAIN), CUDA_PATH=str(TOOLCHAIN),
                  CUDACXX=str(NVCC), CC=str(CC), CXX=str(CXX),
                  CUDAHOSTCXX=str(CXX), MPLBACKEND='Agg', MAX_JOBS='1')
os.environ['PATH'] = str(TOOLCHAIN/'bin') + os.pathsep + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(
    [str(TOOLCHAIN/'lib'), str(TOOLCHAIN/'lib64'), os.environ.get('LD_LIBRARY_PATH', '')])
env = dict(os.environ, PYTHONUNBUFFERED='1')
check_toolchain = """
import os,re,subprocess,torch
from pathlib import Path
from torch.utils.cpp_extension import CUDA_HOME
nvcc=subprocess.check_output([str(Path(CUDA_HOME)/'bin/nvcc'),'--version'],text=True)
gcc=subprocess.check_output([os.environ['CC'],'-dumpfullversion'],text=True).strip()
print('CUDA_HOME=',CUDA_HOME,'torch CUDA=',torch.version.cuda,'GCC=',gcc,flush=True)
print(nvcc,flush=True)
assert re.search(r'release 12\\.1[, ]',nvcc), 'Must compile with CUDA Toolkit 12.1'
assert torch.version.cuda=='12.1', 'Expected the cu121 PyTorch build'
assert gcc.split('.')[0]=='11', 'Expected GCC 11'
"""
run_logged([PY, '-c', check_toolchain], env=env)
# Remove only generated compiler artifacts from the previous CUDA 13 build.
shutil.rmtree(ROOT/'selective_scan/build', ignore_errors=True)
run_logged([PY, '-m', 'pip', 'install', '-vvv', '--no-deps', '--no-build-isolation',
            '--no-cache-dir', str(ROOT/'selective_scan')], env=env)
run_logged([PY, '-c', 'import torch; import selective_scan_cuda_core; import ultralytics; print(ultralytics.__file__); print("PASS: CUDA extension import")'], env=env)
SETUP_OK = True
print('Setup complete. Continue with dataset -> model probe -> training.')


## 2. Chọn phiên bản và dataset
Mặc định T. Có thể chọn B/L nhưng đó là model lớn hơn. Dataset YOLO detection: labels song song images, tọa độ chuẩn hóa. Sửa các giá trị bên dưới sau khi Add Input dataset. Roboflow thường dùng train/images và valid/images.

In [ ]:
import json
VERSION = 'T'  # T / B / L; dùng T để so sánh nhóm model nhỏ
assert VERSION in ('T','B','L')
MODEL = ROOT/f'ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-{VERSION}.yaml'
assert MODEL.is_file(), MODEL
DATA_ROOT = Path('/kaggle/input/YOUR_DATASET')  # SỬA
TRAIN = 'images/train'  # SỬA
VAL = 'images/val'      # SỬA
NAMES = {0:'object'}    # SỬA tất cả tên lớp
assert (DATA_ROOT/TRAIN).is_dir(), DATA_ROOT/TRAIN
assert (DATA_ROOT/VAL).is_dir(), DATA_ROOT/VAL
assert sorted(NAMES)==list(range(len(NAMES))), 'Class ID phải từ 0 liên tiếp'
DATA = Path('/kaggle/working/mamba-data.yaml')
# JSON là YAML hợp lệ, không cần thư viện ngoài trong kernel notebook.
DATA.write_text(json.dumps(dict(path=str(DATA_ROOT),train=TRAIN,val=VAL,names=NAMES),ensure_ascii=False))
EPOCHS = 300  # default launcher của tác giả
BATCH = 4     # điều chỉnh Kaggle; launcher gốc mặc định 512
IMGSZ = 640
SEED = 0
PROJECT = '/kaggle/working/runs-mamba-original'


## 3. Kiểm tra mô hình gốc trước khi train
Không có CPU fallback trong source gốc. Cell này cần CUDA extension hoạt động, nếu lỗi phải dừng.

In [ ]:
assert globals().get('SETUP_OK', False), 'Complete setup successfully before continuing'
probe = """
import sys,torch
from ultralytics import YOLO
model=YOLO(sys.argv[1]).model.cuda().eval()
assert any(type(m).__name__=='VSSBlock' for m in model.modules())
assert any(type(m).__name__=='XSSBlock' for m in model.modules())
with torch.no_grad():
    y=model(torch.rand(2,3,96,96,device='cuda'))
    pred=y[0] if isinstance(y,tuple) else y
    assert torch.isfinite(pred).all()
print('PASS original Mamba architecture:',sum(p.numel() for p in model.parameters()),'parameters')
"""
subprocess.run([PY,'-c',probe,str(MODEL)],check=True)


## 4. Train trực tiếp bằng API gốc
SGD, 300 epochs, lr0/lrf và augment lấy từ default.yaml của source. Chỉ thay batch/device/workers cho Kaggle và dataset của bạn. So sánh với YOLOv12s cần giữ cùng split, epochs, độ phân giải và cách đo; không suy ra model nào tốt hơn nếu chưa train.

In [ ]:
assert globals().get('SETUP_OK', False), 'Complete setup successfully before continuing'
runner = """
import sys,json,subprocess
from pathlib import Path
from ultralytics import YOLO
model_path,data,epochs,batch,imgsz,seed,project,name=sys.argv[1:]
model=YOLO(model_path)
def metadata(trainer):
    import torch
    payload=dict(torch=torch.__version__,cuda=torch.version.cuda,
                 git_commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),
                 model=model_path,original_source=True)
    (trainer.save_dir/'notebook_environment.json').write_text(json.dumps(payload,indent=2))
model.add_callback('on_pretrain_routine_end',metadata)
model.train(data=data,epochs=int(epochs),batch=int(batch),imgsz=int(imgsz),seed=int(seed),
            optimizer='SGD',device='0',workers=2,amp=True,pretrained=False,
            project=project,name=name)
"""
subprocess.run([PY,'-c',runner,str(MODEL),str(DATA),str(EPOCHS),str(BATCH),str(IMGSZ),str(SEED),
                PROJECT,f'Mamba-YOLO-{VERSION}'],check=True)


## 5. Kết quả và tiếp tục run
Save Version kèm outputs. Run trùng tên có hậu tố. Resume dùng args đã lưu; khi đổi phiên Kaggle hãy tạo lại cùng đường dẫn /kaggle/working/mamba-data.yaml ở cell dataset. Chạy setup + dataset rồi resume, không chạy cell train mới.

In [ ]:
for p in sorted(Path(PROJECT).glob('*/weights/*.pt')):
    print(p)
# LAST='/kaggle/input/PREVIOUS_OUTPUT/last.pt'
# subprocess.run([PY,'-c',"import sys; from ultralytics import YOLO; YOLO(sys.argv[1]).train(resume=True)",LAST],check=True)
# BEST='/kaggle/working/runs-mamba-original/Mamba-YOLO-T/weights/best.pt'
# subprocess.run([PY,'-c',"import sys; from ultralytics import YOLO; YOLO(sys.argv[1]).val(data=sys.argv[2],device='0',batch=4)",BEST,str(DATA)],check=True)


Nguồn cài đặt: [NVIDIA CUDA 12.1](https://docs.nvidia.com/cuda/archive/12.1.0/cuda-installation-guide-linux/), [Micromamba](https://mamba.readthedocs.io/en/latest/installation/micromamba-installation.html). CUDA packages lấy từ nvidia/label/cuda-12.1.1; GCC 11 lấy từ conda-forge.
